# 1. Importação dos Dados e Bibliotecas.

Nesta etapa, a biblioteca Pandas é importada e o arquivo de assinaturas é carregado e convertido em um DataFrame.

In [1]:
from pathlib import Path
import sys

RAIZ    = Path.cwd().parent
BRONZE  = RAIZ / "data" / "Bronze"
STAGING = RAIZ / "data" / "Staging"
SILVER  = RAIZ / "data" / "Silver"

assert BRONZE.exists(), f"Pasta não encontrada: {BRONZE}. Abra o notebook a partir da pasta notebooks/."

STAGING.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)
sys.path.append(str(RAIZ))

# Importação de Bibliotecas:
import pandas as pd
import numpy as np

## Importando funções da pasta src.
from src.datas import converter_data
df_pagamentos = pd.read_csv(BRONZE/"pagamentos.csv")
print(df_pagamentos.info())

<class 'pandas.DataFrame'>
RangeIndex: 15020 entries, 0 to 15019
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_pagamento      15020 non-null  int64  
 1   id_assinatura     15020 non-null  int64  
 2   id_cliente        15020 non-null  int64  
 3   data_pagamento    15020 non-null  str    
 4   valor             15020 non-null  float64
 5   status_pagamento  15020 non-null  str    
dtypes: float64(1), int64(3), str(2)
memory usage: 922.3 KB
None


# 2. Tratamento e Análise de Dados:

Nesta etapa, é realizado o tratamento completo dos dados, incluindo a validação de registros duplicados, a alteração de status e a conversão de tipos (types), com base na análise dos seguintes pontos críticos;

In [2]:
# Remoção e Validação de linhas duplicadas dentro de df:
# 1. Validação pré-tratamento: Conta quantas linhas totalmente duplicadas existem na tabela inteira
total_linhas_duplicadas = df_pagamentos.duplicated().sum()
print(f"Total de linhas duplicadas: {total_linhas_duplicadas}")

# 1.2. Tratamento: Remove as linhas duplicadas mantendo apenas a primeira ocorrência
df_pagamentos = df_pagamentos.drop_duplicates(keep='first').reset_index(drop=True)

# 2. Validação pós-tratamento: Nova contagem para garantir que ficou zerado
total_linhas_duplicadas_pos = df_pagamentos.duplicated().sum()
print(f"Total de linhas duplicadas pós tratamento: {total_linhas_duplicadas_pos}")

Total de linhas duplicadas: 20
Total de linhas duplicadas pós tratamento: 0


## 2.1 Tratativa da Coluna id_pagamento:

Nesta etapa será feita a validação de IDs, dados duplicados.

In [3]:
# 1. Validação de IDs duplicados:
total_duplicados = df_pagamentos['id_pagamento'].duplicated().sum()
print(f"Total de IDs duplicados: {total_duplicados}")

# 2. Tratamento de IDs duplicados: mantendo apenas a primeira ocorrência e removendo as repetições a partir da 2ª vez
df_pagamentos = df_pagamentos.drop_duplicates(subset=['id_pagamento'], keep='first')
total_duplicados = df_pagamentos['id_pagamento'].duplicated().sum()

# 3. Validação pós tratamento:
print(f"Total de IDs duplicados pós tratamento: {total_duplicados}")

Total de IDs duplicados: 0
Total de IDs duplicados pós tratamento: 0


## 2.2 Tratativa da Coluna data_pagamento:

Nessa etapa, será feita a transformação da coluna data_pagamento, pois a mesma está com o type int, além disso, será tratado a questão de datas futuras e também as validações dos dados:

In [4]:
# 1. Tratamento de Dados: Transformação do formato de datetime, para padronizar para o estilo iso.
df_pagamentos['data_pagamento'] = converter_data(df_pagamentos['data_pagamento'])

# 1.2 Validação de Dados: Validando dados Pós Tratamento:
print("Tipo data_pagamento:", df_pagamentos['data_pagamento'].dtype)
print("Datas de pagamento vazias:", df_pagamentos['data_pagamento'].isna().sum())


# 2. Tratamento de Datas futuras:
# 2.2 Validação se existem datas futuras.
hoje = pd.Timestamp.today().normalize()
tem_datas_futuras = (df_pagamentos['data_pagamento'] > hoje).any()
print(f'Existem datas posteriores ao dia de hoje ou igual? Resposta: {tem_datas_futuras}')

Tipo data_pagamento: datetime64[us]
Datas de pagamento vazias: 0
Existem datas posteriores ao dia de hoje ou igual? Resposta: False


## 2.3 Tratamento da coluna valor:

Nesta etapa, o objetivo é identificar e corrigir possíveis vendas com valores negativos. Por regra de negócio definida para este dataset, valores negativos não são considerados válidos para pagamentos e serão convertidos para valor absoluto.

In [5]:
# 1. Validação se existem valores negativos:
tem_negativos = (df_pagamentos['valor'] < 0).any()
print(f'Existem valores negativos na coluna valor? Resposta: {tem_negativos}')

# 1.2 Transformando valores em positivos e armazenando valores originais em novas colunas.
df_pagamentos['valor_original'] = df_pagamentos['valor']
df_pagamentos['valor'] = df_pagamentos['valor'].abs()

# 1.3 Validação pós tratamento, caso retorne False, significa que o tratamento foi um sucesso.
tem_negativos_tratado = (df_pagamentos['valor'] < 0).any()
print(f'Existem valores negativos na coluna valor? Resposta: {tem_negativos_tratado}')


Existem valores negativos na coluna valor? Resposta: True
Existem valores negativos na coluna valor? Resposta: False


## 2.4 Tratamento da coluna status_pagamento:

Etapa: Análise, padronização e validação dos valores da coluna 'status'.

In [6]:
# 1. Identificando valores contidos na coluna para tratamento de dados.
valores_unicos = df_pagamentos['status_pagamento'].unique()
print(f'Dados brutos (Sem Tratamento): {valores_unicos}')

# 2. Realizando a correção dos nomes e padronização dos dados:
# 2.1 Removendo todos os espaços extras de dados:
df_pagamentos['status_pagamento'] = df_pagamentos['status_pagamento'].str.strip()

# 2.2 Criação do dicionario de correção:
tipo_permitidos = {
    'Atrasado': 'Atrasado',
    'Pago': 'Pago',
    'Pendente': 'Pendente',
    'Falhou': 'Falhou'
}

# 2.3 Limpeza de Dados: Aplicando o dicionário de correção na minha coluna status_pagamento, para corrigir os valores e manter a padronização.
df_pagamentos['status_pagamento'] = df_pagamentos['status_pagamento'].replace(tipo_permitidos)

# 2.4 Condicional: Caso entre algum valor que não esteja nessa lista, será modificado para "Investigar"
# O .get() procura a chave; se não achar, retorna "Investigar" como padrão
df_pagamentos['status_pagamento'] = df_pagamentos['status_pagamento'].apply(lambda x: tipo_permitidos.get(x, 'Investigar') 
    if x not in ['Atrasado','Pago','Pendente', 'Falhou'] else x)

# 3. Validação de Dados Pós Tratamento:
valores_unicos_tratados = df_pagamentos['status_pagamento'].unique()
print(f'Pós Tratamento: {valores_unicos_tratados}')

Dados brutos (Sem Tratamento): <ArrowStringArray>
['Atrasado', 'Pago', 'Pendente', 'Falhou']
Length: 4, dtype: str
Pós Tratamento: <ArrowStringArray>
['Atrasado', 'Pago', 'Pendente', 'Falhou']
Length: 4, dtype: str


# 3. Data Quality & Validação.

Nesta etapa, o DataFrame tratado é submetido a testes de validação antes de ser promovido para a camada Staging (Preparação de Dados)

In [7]:
resultado_qualidade = {}

# 1. Validaçaõ de IDs não nulos:
resultado_qualidade['ID não nulo'] = (
    df_pagamentos['id_pagamento'].isnull().sum() == 0
)

# 2. Validação a fim de identificar se existem IDs duplicados
resultado_qualidade['ID único'] = (
    df_pagamentos['id_pagamento'].duplicated().sum() == 0
)

# 3. Validação se existem linhas duplicadas
resultado_qualidade['Sem duplicatas completas'] = (
    df_pagamentos.duplicated().sum() == 0
)

# 4. Validação se todas as linhas da coluna id_cliente estão preenchidas
resultado_qualidade['ID cliente preenchido'] = (
    df_pagamentos['id_cliente'].notna().all()
)

# 4.1 Validação se todas as linhas da coluna id_assinatura estão preenchidas
resultado_qualidade['ID assinatura preenchido'] = (
    df_pagamentos['id_assinatura'].notna().all()
)

# 5. Validação se a coluna data_pagamento está com o formato correto de type
resultado_qualidade['Tipo de dado da coluna data_pagamento'] = (
    pd.api.types.is_datetime64_any_dtype(df_pagamentos['data_pagamento'])
)

# 6. Validação se existe campos vazios na coluna data_pagamento
resultado_qualidade['Data pagamento preenchida'] = (
    df_pagamentos['data_pagamento'].notna().all()
)

# 7. Validando se restou algum valor indesejado na coluna status_pagamento:
valores_permitidos_status_pagamento = ['Atrasado','Pago','Pendente', 'Falhou']

# Atribui o resultado True/False diretamente na coluna do dicionário de qualidade
resultado_qualidade['Sem dados instrusos na coluna tipo'] = (
    df_pagamentos['status_pagamento'].isin(valores_permitidos_status_pagamento).all()
)

# 9. Validação de datas futuras, visto que não é possível existir interações com datas futuras.
data_atual = pd.Timestamp.today().normalize()
resultado_qualidade['Sem datas futuras'] = (
    (df_pagamentos['data_pagamento'] <= data_atual).all()
)

# 10. Validação se existe números negativos na coluna 0.
resultado_qualidade['Valor válido'] = (
    (df_pagamentos['valor'] > 0).all()
)

# 11. Validando o type da coluna valor.
resultado_qualidade['Valor numérico'] = (
    pd.api.types.is_numeric_dtype(df_pagamentos['valor'])
)

# 4. Data Quality: Homologação e Envio para a Camada Staging (Preparação de Dados)

Nesta etapa, o console exibe o resultado das validações realizadas:

Aprovado (True em todos os testes): O DataFrame é liberado e enviado para a camada Staging (Preparação de Dados)
Reprovado (qualquer valor False): O processo é bloqueado, exigindo a correção dos dados antes de avançar.

In [8]:
df_qualidade = pd.DataFrame(
    list(resultado_qualidade.items()),
    columns=['Regra', 'Aprovado']
)

df_qualidade['Status'] = df_qualidade['Aprovado'].map({
    True: 'APROVADO',
    False: 'REPROVADO'
})

display(df_qualidade)

todos_testes_ok = df_qualidade['Aprovado'].all()
caminho_saida = (STAGING / "fato_pagamentos.csv")

if todos_testes_ok:
    print("Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging (Preparação de Dados).")

    df_pagamentos.to_csv(
        caminho_saida,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

else:
    print("Alerta: O arquivo foi impedido de subir para a camada Staging (Preparação de Dados). Existem regras reprovadas.")
    raise ValueError("Falha na validação de qualidade dos dados. Processo interrompido.")

,Regra,Aprovado,Status
0,ID não nulo,True,APROVADO
1,ID único,True,APROVADO
2,Sem duplicatas completas,True,APROVADO
3,ID cliente preenchido,True,APROVADO
4,ID assinatura preenchido,True,APROVADO
5,Tipo de dado da coluna data_pagamento,True,APROVADO
6,Data pagamento preenchida,True,APROVADO
7,Sem dados instrusos na coluna tipo,True,APROVADO
8,Sem datas futuras,True,APROVADO
9,Valor válido,True,APROVADO


Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging (Preparação de Dados).
